In [1]:
import numpy as np 
import rasterio
import pandas as pd
import geopandas as gpd
from tqdm import tqdm  
from rasterio.merge import merge
from rasterstats import zonal_stats

## Model 1: AHP weights, ssp2 2025 

In [3]:
# Rasters and weights for model 1 (ssp2 2025):

# Rasters
land_cover = './data/land_cover/land_cover_standardized.tif'
hsg = './data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif'
slope = './data/slope/slope_standardized.tif'
impairment = './data/watershed_impairment/impairment_standardized.tif'
mean_prcp = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif'
sd_prcp = './data/projected_precip/precip_ssp2_2025/ssp2_2025_standard_dev_prcp_rate_standarized.tif'
impervious = './data/percent_impervious/impervious_standardized.tif'
pop_density = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif'
svi = './data/svi/svi_standardized.tif'

# Weights
land_cover_weight = 0.2817
hsg_weight = 0.2039
slope_weight = 0.2005
impairment_weight = 0.1078
mean_prcp_weight = 0.0567
sd_prcp_weight = 0.0567
impervious_weight = 0.0567
pop_density_weight = 0.0211
svi_weight = 0.015

in_rasters = [land_cover, 
              hsg, slope, 
              impairment, 
              mean_prcp, 
              sd_prcp, 
              impervious, 
              pop_density, 
              svi]

weights = [land_cover_weight, 
           hsg_weight, 
           slope_weight, 
           impairment_weight, 
           mean_prcp_weight, 
           sd_prcp_weight, 
           impervious_weight, 
           pop_density_weight, 
           svi_weight]

for raster, weight in zip(in_rasters, weights):
    print(raster, weight)

./data/land_cover/land_cover_standardized.tif 0.2817
./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif 0.2039
./data/slope/slope_standardized.tif 0.2005
./data/watershed_impairment/impairment_standardized.tif 0.1078
./data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif 0.0567
./data/projected_precip/precip_ssp2_2025/ssp2_2025_standard_dev_prcp_rate_standarized.tif 0.0567
./data/percent_impervious/impervious_standardized.tif 0.0567
./data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif 0.0211
./data/svi/svi_standardized.tif 0.015


In [7]:
# Model 1 weighted overlay 

# Output
out_raster = './data/final_outputs/model1_ahp_weights_ssp2_2025_TWO.tif'

# Open all rasters
srcs = [rasterio.open(raster) for raster in in_rasters]

# Verify that there are the same number of srcs as there are weights
print(f'Lengh of srcs equals length of weights: {len(srcs) == len(weights)}')
for src, weight in zip(srcs, weights):
    print(src, weight)

# Copy metadata from src raster
meta = srcs[0].meta.copy()
nodata = srcs[0].nodata

meta.update(
    dtype = rasterio.float32,
    count = 1,
    nodata = nodata,
    tiled = True, 
    blockxsize = 128, 
    blockysize = 128,
    compress = 'deflate',
    predictor = 3,
    BIGTIFF = 'yes'
)

# Total blocks
total_blocks = sum(1 for _ in srcs[0].block_windows(1))
print(f'Total blocks {total_blocks}')

# Make output raster
with rasterio.open(out_raster, 'w', **meta) as dst:
    
    # For each block window
    for ji, window in tqdm(srcs[0].block_windows(1), total = total_blocks, desc = 'Block window processing'):
        # Create an empty array of zeros
        weighted_sum = np.zeros(
            (window.height, window.width),
            dtype = rasterio.float32)

        # Create an array to store all VALID (aka not nodata) cells (starts off with all cells as valid)
        valid_array = np.ones(
            (window.height, window.width),
            dtype = bool)
        
        # Go through each variable, weighs
        for src, weight in zip(srcs, weights):
            data = src.read(1, window = window)

            # Valid cells (where the cells do not equal nodata)
            valid_cells = data!= nodata

            # Update valid_array after checking for nodata cells
            valid_array = valid_array & valid_cells # Only True if the cell is actually valid

            # Replace nodata cells with 0 to not mess up with weighted sum calculations
            data = np.where(valid_cells, data, 0)

            # Multiply the cell value by the weight 
            weighted_sum += data*weight

        # Set any nodata cells to nodata in the final output raster
        weighted_sum = np.where(valid_array, weighted_sum, nodata)

        # Write out final raster
        dst.write(weighted_sum, 1, window = window)
            

# Close all input rasters
for src in srcs:
    src.close()

Lengh of srcs equals length of weights: True
<open DatasetReader name='./data/land_cover/land_cover_standardized.tif' mode='r'> 0.2817
<open DatasetReader name='./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif' mode='r'> 0.2039
<open DatasetReader name='./data/slope/slope_standardized.tif' mode='r'> 0.2005
<open DatasetReader name='./data/watershed_impairment/impairment_standardized.tif' mode='r'> 0.1078
<open DatasetReader name='./data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/projected_precip/precip_ssp2_2025/ssp2_2025_standard_dev_prcp_rate_standarized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/percent_impervious/impervious_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif' mode='r'> 0.0211
<open DatasetReader name='./data/svi/svi_standardized.tif' mode='r'> 0.015
Total blocks 910224


Block window processing: 100%|██████████| 910224/910224 [1:01:35<00:00, 246.28it/s]


In [8]:
# Model 1 output raster

out_raster = './data/final_outputs/model1_ahp_weights_ssp2_2025_TWO.tif'

with rasterio.open(out_raster, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [11]:
# Reclassify cells for Model 1

out_raster = './data/final_outputs/model1_ahp_weights_ssp2_2025_TWO.tif'
out_raster_reclassified = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif'

with rasterio.open(out_raster) as src:
    profile = src.profile.copy()
    nodata = src.nodata
    profile.update(
        dtype = rasterio.int16,
        nodata = nodata,
        #blockxsize = 128,
        #blockysize = 128,
        #tiled = True
    )

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    with rasterio.open(out_raster_reclassified, 'w', **profile) as dst:

        for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):

            data = src.read(1, window = window)
            data_copy = data.copy()

            # Reclassify:
                # [10-8] >> 5 (highly suitable)
            data_copy[(data <=10) & (data >=8)] = 5
            
                # (8-6] >> 4 (moderately suitable)
            data_copy[(data <8) & (data >=6)] = 4
            
                # (6-4}] >> 3 (marginally suitable)
            data_copy[(data <6) & (data >=4)] = 3

                # (4-2] >> 2 (very unsuitable)
            data_copy[(data <4) & (data >=2)] = 2

                # (2-0] >> 1 (unsuitable)
            data_copy[(data <2) & (data >=0)] = 1

            # Write out reclassified raster
            dst.write(data_copy, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [14:27<00:00, 1049.67it/s]


In [12]:
# Model 1 reclassified output raster

out_raster_reclassified = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif'

with rasterio.open(out_raster_reclassified, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso

In [13]:
# Calculate % of each final suitability category

out_raster_reclassified = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif'

total_1 = 0
total_2 = 0
total_3 = 0
total_4 = 0
total_5 = 0
total_nodata = 0

with rasterio.open(out_raster_reclassified) as src:

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
        data = src.read(1, window = window, masked = True)

        total_1 += np.count_nonzero(data == 1) # Total cells = 1
        total_2 += np.count_nonzero(data == 2) # Total cells = 2
        total_3 += np.count_nonzero(data == 3) # Total cells = 3
        total_4 += np.count_nonzero(data == 4) # Total cells = 4
        total_5 += np.count_nonzero(data == 5) # Total cells = 5

        total_nodata += np.sum(data.mask) # Calculates the total number of nodata cells

    all_cells = src.width * src.height

    all_NON_nodata_cells = all_cells - total_nodata

    percent_5 = total_5 / all_NON_nodata_cells
    print(f'% highly suitable (5): {percent_5}')
    
    percent_4 = total_4 / all_NON_nodata_cells
    print(f'% moderately suitable (4): {percent_4}')

    percent_3 = total_3 / all_NON_nodata_cells
    print(f'% marginally suitable (3): {percent_3}')

    percent_2 = total_2 / all_NON_nodata_cells
    print(f'% very unsuitable (2): {percent_2}')

    percent_1 = total_1 / all_NON_nodata_cells
    print(f'% absolutely unsuitable (1): {percent_1}')

    print(f'%s add to : {percent_1 + percent_2 + percent_3 + percent_4 + percent_5}')

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:08<00:00, 2951.54it/s]

% highly suitable (5): 0.014563700070792782
% moderately suitable (4): 0.07682775207877833
% marginally suitable (3): 0.6615352921843293
% very unsuitable (2): 0.23401168832776678
% absolutely unsuitable (1): 0.013061567338332846
%s add to : 1.0


## Model 2: AHP weights, ssp2 2050 

In [2]:
# Rasters and weights for model 2 (ssp2 2050):

# Rasters
land_cover = './data/land_cover/land_cover_standardized.tif'
hsg = './data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif'
slope = './data/slope/slope_standardized.tif'
impairment = './data/watershed_impairment/impairment_standardized.tif'
mean_prcp = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_standardized.tif' # Change
sd_prcp = './data/projected_precip/precip_ssp2_2050/ssp2_2050_standard_dev_prcp_rate_standardized.tif' # Change
impervious = './data/percent_impervious/impervious_standardized.tif'
pop_density = './data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_standardized.tif' # Change
svi = './data/svi/svi_standardized.tif'

# Weights
land_cover_weight = 0.2817
hsg_weight = 0.2039
slope_weight = 0.2005
impairment_weight = 0.1078
mean_prcp_weight = 0.0567
sd_prcp_weight = 0.0567
impervious_weight = 0.0567
pop_density_weight = 0.0211
svi_weight = 0.015

in_rasters = [land_cover, 
              hsg, slope, 
              impairment, 
              mean_prcp, 
              sd_prcp, 
              impervious, 
              pop_density, 
              svi]

weights = [land_cover_weight, 
           hsg_weight, 
           slope_weight, 
           impairment_weight, 
           mean_prcp_weight, 
           sd_prcp_weight, 
           impervious_weight, 
           pop_density_weight, 
           svi_weight]

for raster, weight in zip(in_rasters, weights):
    print(raster, weight)

./data/land_cover/land_cover_standardized.tif 0.2817
./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif 0.2039
./data/slope/slope_standardized.tif 0.2005
./data/watershed_impairment/impairment_standardized.tif 0.1078
./data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_standardized.tif 0.0567
./data/projected_precip/precip_ssp2_2050/ssp2_2050_standard_dev_prcp_rate_standardized.tif 0.0567
./data/percent_impervious/impervious_standardized.tif 0.0567
./data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_standardized.tif 0.0211
./data/svi/svi_standardized.tif 0.015


In [3]:
# Model 2 weighted overlay 

# Output
out_raster = './data/final_outputs/model2_ahp_weights_ssp2_2050.tif'

# Open all rasters
srcs = [rasterio.open(raster) for raster in in_rasters]

# Verify that there are the same number of srcs as there are weights
print(f'Lengh of srcs equals length of weights: {len(srcs) == len(weights)}')
for src, weight in zip(srcs, weights):
    print(src, weight)

# Copy metadata from src raster
meta = srcs[0].meta.copy()
nodata = srcs[0].nodata

meta.update(
    dtype = rasterio.float32,
    count = 1,
    nodata = nodata,
    tiled = True, 
    blockxsize = 128, 
    blockysize = 128,
    compress = 'deflate',
    predictor = 3,
    BIGTIFF = 'yes'
)

# Total blocks
total_blocks = sum(1 for _ in srcs[0].block_windows(1))
print(f'Total blocks {total_blocks}')

# Make output raster
with rasterio.open(out_raster, 'w', **meta) as dst:
    
    # For each block window
    for ji, window in tqdm(srcs[0].block_windows(1), total = total_blocks, desc = 'Block window processing'):
        # Create an empty array of zeros
        weighted_sum = np.zeros(
            (window.height, window.width),
            dtype = rasterio.float32)

        # Create an array to store all VALID (aka not nodata) cells (starts off with all cells as valid)
        valid_array = np.ones(
            (window.height, window.width),
            dtype = bool)
        
        # Go through each variable, weighs
        for src, weight in zip(srcs, weights):
            data = src.read(1, window = window)

            # Valid cells (where the cells do not equal nodata)
            valid_cells = data!= nodata

            # Update valid_array after checking for nodata cells
            valid_array = valid_array & valid_cells # Only True if the cell is actually valid

            # Replace nodata cells with 0 to not mess up with weighted sum calculations
            data = np.where(valid_cells, data, 0)

            # Multiply the cell value by the weight 
            weighted_sum += data*weight

        # Set any nodata cells to nodata in the final output raster
        weighted_sum = np.where(valid_array, weighted_sum, nodata)

        # Write out final raster
        dst.write(weighted_sum, 1, window = window)
            

# Close all input rasters
for src in srcs:
    src.close()

Lengh of srcs equals length of weights: True
<open DatasetReader name='./data/land_cover/land_cover_standardized.tif' mode='r'> 0.2817
<open DatasetReader name='./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif' mode='r'> 0.2039
<open DatasetReader name='./data/slope/slope_standardized.tif' mode='r'> 0.2005
<open DatasetReader name='./data/watershed_impairment/impairment_standardized.tif' mode='r'> 0.1078
<open DatasetReader name='./data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/projected_precip/precip_ssp2_2050/ssp2_2050_standard_dev_prcp_rate_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/percent_impervious/impervious_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/population_density/pop_ssp2_rcp45_2050/ssp2_2050_density_change_standardized.tif' mode='r'> 0.0211
<open DatasetReader name='./data/svi/svi_standardized.tif' mode='r'> 0.015
Total blocks 910224


Block window processing: 100%|██████████| 910224/910224 [1:03:15<00:00, 239.81it/s]


In [4]:
# Model 2 output raster

out_raster = './data/final_outputs/model2_ahp_weights_ssp2_2050.tif'

with rasterio.open(out_raster, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [14]:
# Reclassify cells for Model 2

out_raster = './data/final_outputs/model2_ahp_weights_ssp2_2050.tif'
out_raster_reclassified = './data/final_outputs/model2_ahp_weights_ssp2_2050_RECLASSIFIED.tif'

with rasterio.open(out_raster) as src:
    profile = src.profile.copy()
    nodata = src.nodata
    profile.update(
        dtype = rasterio.int16,
        nodata = nodata,
        #blockxsize = 128,
        #blockysize = 128,
        #tiled = True
    )

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    with rasterio.open(out_raster_reclassified, 'w', **profile) as dst:

        for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):

            data = src.read(1, window = window)
            data_copy = data.copy()

            # Reclassify:
                # [10-8] >> 5 (highly suitable)
            data_copy[(data <=10) & (data >=8)] = 5
            
                # (8-6] >> 4 (moderately suitable)
            data_copy[(data <8) & (data >=6)] = 4
            
                # (6-4}] >> 3 (marginally suitable)
            data_copy[(data <6) & (data >=4)] = 3

                # (4-2] >> 2 (very unsuitable)
            data_copy[(data <4) & (data >=2)] = 2

                # (2-0] >> 1 (unsuitable)
            data_copy[(data <2) & (data >=0)] = 1

            # Write out reclassified raster
            dst.write(data_copy, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [14:23<00:00, 1054.42it/s]


In [15]:
# Model 2 reclassified output raster

out_raster_reclassified = './data/final_outputs/model2_ahp_weights_ssp2_2050_RECLASSIFIED.tif'

with rasterio.open(out_raster_reclassified, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso

In [16]:
# Calculate % of each final suitability category

out_raster_reclassified = './data/final_outputs/model2_ahp_weights_ssp2_2050_RECLASSIFIED.tif'

total_1 = 0
total_2 = 0
total_3 = 0
total_4 = 0
total_5 = 0
total_nodata = 0

with rasterio.open(out_raster_reclassified) as src:

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
        data = src.read(1, window = window, masked = True)

        total_1 += np.count_nonzero(data == 1) # Total cells = 1
        total_2 += np.count_nonzero(data == 2) # Total cells = 2
        total_3 += np.count_nonzero(data == 3) # Total cells = 3
        total_4 += np.count_nonzero(data == 4) # Total cells = 4
        total_5 += np.count_nonzero(data == 5) # Total cells = 5

        total_nodata += np.sum(data.mask) # Calculates the total number of nodata cells

    all_cells = src.width * src.height

    all_NON_nodata_cells = all_cells - total_nodata

    percent_5 = total_5 / all_NON_nodata_cells
    print(f'% highly suitable (5): {percent_5}')
    
    percent_4 = total_4 / all_NON_nodata_cells
    print(f'% moderately suitable (4): {percent_4}')

    percent_3 = total_3 / all_NON_nodata_cells
    print(f'% marginally suitable (3): {percent_3}')

    percent_2 = total_2 / all_NON_nodata_cells
    print(f'% very unsuitable (2): {percent_2}')

    percent_1 = total_1 / all_NON_nodata_cells
    print(f'% absolutely unsuitable (1): {percent_1}')

    print(f'%s add to : {percent_1 + percent_2 + percent_3 + percent_4 + percent_5}')

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:17<00:00, 2863.86it/s]


% highly suitable (5): 0.014153156380270157
% moderately suitable (4): 0.07801233543814322
% marginally suitable (3): 0.6621182628749088
% very unsuitable (2): 0.23271814547562786
% absolutely unsuitable (1): 0.012998099831049902
%s add to : 1.0


## Model 3: AHP weights, ssp2 2050 

In [5]:
# Rasters and weights for model 3 (ssp5 2050):

# Rasters
land_cover = './data/land_cover/land_cover_standardized.tif'
hsg = './data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif'
slope = './data/slope/slope_standardized.tif'
impairment = './data/watershed_impairment/impairment_standardized.tif'
mean_prcp = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_standardized.tif' # Change
sd_prcp = './data/projected_precip/precip_ssp5_2050/ssp5_2050_standard_dev_prcp_rate_standardized.tif' # Change
impervious = './data/percent_impervious/impervious_standardized.tif'
pop_density = './data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_standardized.tif' # Change
svi = './data/svi/svi_standardized.tif'

# Weights
land_cover_weight = 0.2817
hsg_weight = 0.2039
slope_weight = 0.2005
impairment_weight = 0.1078
mean_prcp_weight = 0.0567
sd_prcp_weight = 0.0567
impervious_weight = 0.0567
pop_density_weight = 0.0211
svi_weight = 0.015

in_rasters = [land_cover, 
              hsg, slope, 
              impairment, 
              mean_prcp, 
              sd_prcp, 
              impervious, 
              pop_density, 
              svi]

weights = [land_cover_weight, 
           hsg_weight, 
           slope_weight, 
           impairment_weight, 
           mean_prcp_weight, 
           sd_prcp_weight, 
           impervious_weight, 
           pop_density_weight, 
           svi_weight]

for raster, weight in zip(in_rasters, weights):
    print(raster, weight)

./data/land_cover/land_cover_standardized.tif 0.2817
./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif 0.2039
./data/slope/slope_standardized.tif 0.2005
./data/watershed_impairment/impairment_standardized.tif 0.1078
./data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_standardized.tif 0.0567
./data/projected_precip/precip_ssp5_2050/ssp5_2050_standard_dev_prcp_rate_standardized.tif 0.0567
./data/percent_impervious/impervious_standardized.tif 0.0567
./data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_standardized.tif 0.0211
./data/svi/svi_standardized.tif 0.015


In [6]:
# Model 3 weighted overlay 

# Output
out_raster = './data/final_outputs/model3_ahp_weights_ssp5_2050.tif'

# Open all rasters
srcs = [rasterio.open(raster) for raster in in_rasters]

# Verify that there are the same number of srcs as there are weights
print(f'Lengh of srcs equals length of weights: {len(srcs) == len(weights)}')
for src, weight in zip(srcs, weights):
    print(src, weight)

# Copy metadata from src raster
meta = srcs[0].meta.copy()
nodata = srcs[0].nodata

meta.update(
    dtype = rasterio.float32,
    count = 1,
    nodata = nodata,
    tiled = True, 
    blockxsize = 128, 
    blockysize = 128,
    compress = 'deflate',
    predictor = 3,
    BIGTIFF = 'yes'
)

# Total blocks
total_blocks = sum(1 for _ in srcs[0].block_windows(1))
print(f'Total blocks {total_blocks}')

# Make output raster
with rasterio.open(out_raster, 'w', **meta) as dst:
    
    # For each block window
    for ji, window in tqdm(srcs[0].block_windows(1), total = total_blocks, desc = 'Block window processing'):
        # Create an empty array of zeros
        weighted_sum = np.zeros(
            (window.height, window.width),
            dtype = rasterio.float32)

        # Create an array to store all VALID (aka not nodata) cells (starts off with all cells as valid)
        valid_array = np.ones(
            (window.height, window.width),
            dtype = bool)
        
        # Go through each variable, weighs
        for src, weight in zip(srcs, weights):
            data = src.read(1, window = window)

            # Valid cells (where the cells do not equal nodata)
            valid_cells = data!= nodata

            # Update valid_array after checking for nodata cells
            valid_array = valid_array & valid_cells # Only True if the cell is actually valid

            # Replace nodata cells with 0 to not mess up with weighted sum calculations
            data = np.where(valid_cells, data, 0)

            # Multiply the cell value by the weight 
            weighted_sum += data*weight

        # Set any nodata cells to nodata in the final output raster
        weighted_sum = np.where(valid_array, weighted_sum, nodata)

        # Write out final raster
        dst.write(weighted_sum, 1, window = window)
            

# Close all input rasters
for src in srcs:
    src.close()

Lengh of srcs equals length of weights: True
<open DatasetReader name='./data/land_cover/land_cover_standardized.tif' mode='r'> 0.2817
<open DatasetReader name='./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif' mode='r'> 0.2039
<open DatasetReader name='./data/slope/slope_standardized.tif' mode='r'> 0.2005
<open DatasetReader name='./data/watershed_impairment/impairment_standardized.tif' mode='r'> 0.1078
<open DatasetReader name='./data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/projected_precip/precip_ssp5_2050/ssp5_2050_standard_dev_prcp_rate_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/percent_impervious/impervious_standardized.tif' mode='r'> 0.0567
<open DatasetReader name='./data/population_density/pop_ssp5_rcp85_2050/ssp5_2050_density_change_standardized.tif' mode='r'> 0.0211
<open DatasetReader name='./data/svi/svi_standardized.tif' mode='r'> 0.015
Total blocks 910224


Block window processing: 100%|██████████| 910224/910224 [1:02:35<00:00, 242.38it/s]


In [7]:
# Model 3 output raster

out_raster = './data/final_outputs/model3_ahp_weights_ssp5_2050.tif'

with rasterio.open(out_raster, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [17]:
# Reclassify cells for Model 3

out_raster = './data/final_outputs/model3_ahp_weights_ssp5_2050.tif'
out_raster_reclassified = './data/final_outputs/model3_ahp_weights_ssp5_2050_RECLASSIFIED.tif'

with rasterio.open(out_raster) as src:
    profile = src.profile.copy()
    nodata = src.nodata
    profile.update(
        dtype = rasterio.int16,
        nodata = nodata,
        #blockxsize = 128,
        #blockysize = 128,
        #tiled = True
    )

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    with rasterio.open(out_raster_reclassified, 'w', **profile) as dst:

        for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):

            data = src.read(1, window = window)
            data_copy = data.copy()

            # Reclassify:
                # [10-8] >> 5 (highly suitable)
            data_copy[(data <=10) & (data >=8)] = 5
            
                # (8-6] >> 4 (moderately suitable)
            data_copy[(data <8) & (data >=6)] = 4
            
                # (6-4}] >> 3 (marginally suitable)
            data_copy[(data <6) & (data >=4)] = 3

                # (4-2] >> 2 (very unsuitable)
            data_copy[(data <4) & (data >=2)] = 2

                # (2-0] >> 1 (unsuitable)
            data_copy[(data <2) & (data >=0)] = 1

            # Write out reclassified raster
            dst.write(data_copy, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [14:03<00:00, 1078.72it/s]


In [18]:
# Model 3 reclassified output raster

out_raster_reclassified = './data/final_outputs/model3_ahp_weights_ssp5_2050_RECLASSIFIED.tif'

with rasterio.open(out_raster_reclassified, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso

In [19]:
# Calculate % of each final suitability category

out_raster_reclassified = './data/final_outputs/model3_ahp_weights_ssp5_2050_RECLASSIFIED.tif'

total_1 = 0
total_2 = 0
total_3 = 0
total_4 = 0
total_5 = 0
total_nodata = 0

with rasterio.open(out_raster_reclassified) as src:

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
        data = src.read(1, window = window, masked = True)

        total_1 += np.count_nonzero(data == 1) # Total cells = 1
        total_2 += np.count_nonzero(data == 2) # Total cells = 2
        total_3 += np.count_nonzero(data == 3) # Total cells = 3
        total_4 += np.count_nonzero(data == 4) # Total cells = 4
        total_5 += np.count_nonzero(data == 5) # Total cells = 5

        total_nodata += np.sum(data.mask) # Calculates the total number of nodata cells

    all_cells = src.width * src.height

    all_NON_nodata_cells = all_cells - total_nodata

    percent_5 = total_5 / all_NON_nodata_cells
    print(f'% highly suitable (5): {percent_5}')
    
    percent_4 = total_4 / all_NON_nodata_cells
    print(f'% moderately suitable (4): {percent_4}')

    percent_3 = total_3 / all_NON_nodata_cells
    print(f'% marginally suitable (3): {percent_3}')

    percent_2 = total_2 / all_NON_nodata_cells
    print(f'% very unsuitable (2): {percent_2}')

    percent_1 = total_1 / all_NON_nodata_cells
    print(f'% absolutely unsuitable (1): {percent_1}')

    print(f'%s add to : {percent_1 + percent_2 + percent_3 + percent_4 + percent_5}')

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:09<00:00, 2940.55it/s]

% highly suitable (5): 0.01596726008098344
% moderately suitable (4): 0.08209912616112823
% marginally suitable (3): 0.6596371777230488
% very unsuitable (2): 0.2293497221303464
% absolutely unsuitable (1): 0.012946713904493142
%s add to : 1.0


## Combining GSI and urban Culex spp. rasters

In [2]:
# Reclassify urban Culex spp. raster to use for combination with GSI rasters

urban_culex_reclassified_suitability = './data/mosquitoes/urban_culex_composite_reclassified_suitability_pip_quin.tif'
urban_culex_reclassified_gsi_mos = './data/final_outputs/intermediate_reclassified_rasters/urban_culex_composite_reclassified_suitability_for_combining.tif'
    
with rasterio.open(urban_culex_reclassified_suitability, mode = 'r') as src:
    profile = src.profile.copy()

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')
    
    with rasterio.open(urban_culex_reclassified_gsi_mos, 'w', **profile) as dst:
        for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
            data = src.read(1, window = window)
            
            # Convert 5 >> 100
            data[data == 5] = 100
            
            # Convert 4 >> 200
            data[data == 4] = 200
            
            # Convert 3 >> 25
            data[data == 3] = 25
            
            # Convert 2 >> 25
            data[data == 2] = 25

            # Convert 1 >> 25
            data[data == 1] = 25
            
            # Write out new raster
            dst.write(data, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:21<00:00, 2834.94it/s]


In [3]:
# Reclassify model 1 GSI output raster to use for combination with GSI rasters

model1_gsi = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif'
model1_gsi_reclassified_gsi_mos = './data/final_outputs/intermediate_reclassified_rasters/model1_reclassified_intermediary.tif'
    
with rasterio.open(model1_gsi, mode = 'r') as src:
    profile = src.profile.copy()

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')
    
    with rasterio.open(model1_gsi_reclassified_gsi_mos, 'w', **profile) as dst:
        for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
            data = src.read(1, window = window)
            
            # Convert 5 >> 10
            data[data == 5] = 10
            
            # Convert 4 >> 20
            data[data == 4] = 20
            
            # Convert 3 >> 25
            data[data == 3] = 25
            
            # Convert 2 >> 25
            data[data == 2] = 25

            # Convert 1 >> 25
            data[data == 1] = 25
            
            # Write out new raster
            dst.write(data, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [06:23<00:00, 2376.18it/s]


In [4]:
# Function to check profiles match
def raster_profile_match(raster1, raster2):
    with rasterio.open(raster1) as src1:
        profile1 = src1.profile
    with rasterio.open(raster2) as src2:
        profile2 = src2.profile
    if profile1 == profile2:
        print('Raster profiles are identical')
    else:
        print('Raster profiles are not identical')

# Raster profiles to check
urban_culex_reclassified_gsi_mos = './data/final_outputs/intermediate_reclassified_rasters/urban_culex_composite_reclassified_suitability_for_combining.tif'
model1_gsi_reclassified_gsi_mos = './data/final_outputs/intermediate_reclassified_rasters/model1_reclassified_intermediary.tif'

raster_profile_match(urban_culex_reclassified_gsi_mos, model1_gsi_reclassified_gsi_mos)

Raster profiles are identical


In [5]:
# Sum rasters

urban_culex_reclassified_gsi_mos = './data/final_outputs/intermediate_reclassified_rasters/urban_culex_composite_reclassified_suitability_for_combining.tif'
model1_gsi_reclassified_gsi_mos = './data/final_outputs/intermediate_reclassified_rasters/model1_reclassified_intermediary.tif'

gsi_culex_model1 = './data/final_outputs/model1_gsi_mosquitoes_composite.tif'


with rasterio.open(urban_culex_reclassified_gsi_mos) as src1:
    profile = src1.profile.copy()
    nodata = src1.nodata

    # Total number of blocks
    total_blocks = sum(1 for _ in src1.block_windows(1))
    print(f'Total blocks: {total_blocks}')
    
    with rasterio.open(model1_gsi_reclassified_gsi_mos) as src2:

        with rasterio.open(gsi_culex_model1, 'w', **profile) as dst:

            for ji, window in tqdm(src1.block_windows(1), total = total_blocks, desc = 'Block window processing'):
                culex_data = src1.read(1, window = window) # Read in culex data
                gsi_data = src2.read(1, window = window) # Read in GSI data

                summed = culex_data + gsi_data # Sum rasters

                nodata_cells = (culex_data == nodata) | (gsi_data == nodata) # Isolate nodata cells
                summed[nodata_cells] = nodata # Replace nodata cells with nodata value

                dst.write(summed, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [07:32<00:00, 2009.58it/s]


In [6]:
# View final raster combining gsi and culex suitability - model 1

gsi_culex_model1 = './data/final_outputs/model1_gsi_mosquitoes_composite.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(gsi_culex_model1, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso

In [7]:
# Reclassify to FINAL composite scores - change all unsuitable combinations to 25

gsi_culex_model1 = './data/final_outputs/model1_gsi_mosquitoes_composite.tif'
gsi_culex_model1_FINAL_reclass = './data/final_outputs/model1_gsi_mosquitoes_composite_FINAL_reclass.tif'
    
with rasterio.open(gsi_culex_model1, mode = 'r') as src:
    profile = src.profile.copy()

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')
    
    with rasterio.open(gsi_culex_model1_FINAL_reclass, 'w', **profile) as dst:
        for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
            data = src.read(1, window = window)
            
            # Cells with the following values indicate unsuitability for GSI, culex, or both >> reclassify to 25
            data[(data == 125)|(data == 225)|(data == 45)|(data == 35)] = 50
            
            # Write out new raster
            dst.write(data, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:06<00:00, 2973.01it/s]


In [8]:
# View final raster combining gsi and culex suitability - model 1 - FINAL COMPOSITE

gsi_culex_model1_FINAL_reclass = './data/final_outputs/model1_gsi_mosquitoes_composite_FINAL_reclass.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(gsi_culex_model1_FINAL_reclass, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso

In [9]:
# Calculate % of each final suitability category

gsi_culex_model1_FINAL_reclass = './data/final_outputs/model1_gsi_mosquitoes_composite_FINAL_reclass.tif'

total_50 = 0
total_110 = 0
total_120 = 0
total_210 = 0
total_220 = 0
total_nodata = 0

with rasterio.open(gsi_culex_model1_FINAL_reclass) as src:

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
        data = src.read(1, window = window, masked = True)

        total_50 += np.count_nonzero(data == 50) # Total cells = 50
        total_110 += np.count_nonzero(data == 110) # Total cells = 110
        total_120 += np.count_nonzero(data == 120) # Total cells = 120
        total_210 += np.count_nonzero(data == 210) # Total cells = 210
        total_220 += np.count_nonzero(data == 220) # Total cells = 220

        total_nodata += np.sum(data.mask) # Calculates the total number of nodata cells

    all_cells = src.width * src.height

    all_NON_nodata_cells = all_cells - total_nodata

    percent_50 = total_50 / all_NON_nodata_cells
    print(f'% unsuitable for either gsi or culex (50): {percent_50}')
    
    percent_110 = total_110 / all_NON_nodata_cells
    print(f'% high culex, high gsi suitability (110): {percent_110}')

    percent_120 = total_120 / all_NON_nodata_cells
    print(f'% high culex, mod gsi suitability (120): {percent_120}')

    percent_210 = total_210 / all_NON_nodata_cells
    print(f'% mod culex, high gsi suitability: {percent_210}')

    percent_220 = total_220 / all_NON_nodata_cells
    print(f'% mod culex, mod gsi suitability: {percent_220}')

    print(f'%s add to : {percent_50 + percent_110 + percent_120 + percent_210 + percent_220}')

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:15<00:00, 2884.82it/s]


% unsuitable for either gsi or culex (50): 0.9495233598195552
% high culex, high gsi suitability (110): 0.008724834653697992
% high culex, mod gsi suitability (120): 0.019130068470418832
% mod culex, high gsi suitability: 0.0030638553855463907
% mod culex, mod gsi suitability: 0.01955788167078167
%s add to : 1.0000000000000002


In [12]:
# % of cells highly suitable for GSI that are also highly suitable for Culex (CONUS)
gsi_highly_suitable = 0
highly_suitabile_BOTH = 0

gsi_suitability_conus = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif'
culex_suitability_conus = './data/mosquitoes/urban_culex_composite_reclassified_suitability_pip_quin.tif'

with rasterio.open(gsi_suitability_conus) as gsi, rasterio.open(culex_suitability_conus) as culex:
    for ji, window in gsi.block_windows(1):
        gsi_data = gsi.read(1, window = window, masked = True)
        culex_data = culex.read(1, window = window, masked = True)

        combined_mask = gsi_data.mask | culex_data.mask
        gsi_data = np.ma.array(gsi_data.data, mask=combined_mask)
        culex_data = np.ma.array(culex_data.data, mask=combined_mask)
        
        gsi_highly_suitable += np.count_nonzero(gsi_data == 5) # Total number of cells highly suitable for GSI
        highly_suitabile_BOTH += np.count_nonzero((gsi_data == 5) & (culex_data == 5))

pct = highly_suitabile_BOTH / gsi_highly_suitable * 100
print(f'(Number of cells highly suitable for GSI: {gsi_highly_suitable}')
print(f'(Number of cells highly suitable for Culex & GSI: {highly_suitabile_BOTH}')
print(f'(Percent of cells highly suitable for GSI that are also highly suitable for Culex: {pct}')

(Number of cells highly suitable for GSI: 124628821
(Number of cells highly suitable for Culex & GSI: 74703490
(Percent of cells highly suitable for GSI that are also highly suitable for Culex: 59.9407820764027


In [13]:
# % of cells highly or moderately suitable for GSI that are also highly or moderately suitable for Culex (CONUS)
total_gsi_high_mod_suitable = 0
total_high_mod_suitable_BOTH = 0

gsi_suitability_conus = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif'
culex_suitability_conus = './data/mosquitoes/urban_culex_composite_reclassified_suitability_pip_quin.tif'

with rasterio.open(gsi_suitability_conus) as gsi, rasterio.open(culex_suitability_conus) as culex:
    for ji, window in gsi.block_windows(1):
        gsi_data = gsi.read(1, window = window, masked = True)
        culex_data = culex.read(1, window = window, masked = True)

        combined_mask = gsi_data.mask | culex_data.mask
        gsi_data = np.ma.array(gsi_data.data, mask=combined_mask)
        culex_data = np.ma.array(culex_data.data, mask=combined_mask)

        gsi_high_mod = (gsi_data == 5) | (gsi_data == 4) # Cells highly or moderately suitable for GSI
        culex_high_mod = (culex_data == 5) | (culex_data == 4) # Cells highly or moderately suitable for Culex
        
        total_gsi_high_mod_suitable += np.count_nonzero(gsi_high_mod) # Total number of cells highly or moderately suitable for GSI
        total_high_mod_suitable_BOTH += np.count_nonzero(gsi_high_mod & culex_high_mod)

pct = total_high_mod_suitable_BOTH / total_gsi_high_mod_suitable * 100
print(f'(Number of cells highly or moderately suitable for GSI: {total_gsi_high_mod_suitable}')
print(f'(Number of cells highly suitable for Culex & GSI: {total_high_mod_suitable_BOTH}')
print(f'(Percent of cells highly suitable for GSI that are also highly suitable for Culex: {pct}')

(Number of cells highly or moderately suitable for GSI: 782463271
(Number of cells highly suitable for Culex & GSI: 432189415
(Percent of cells highly suitable for GSI that are also highly suitable for Culex: 55.23446671786336


## Model 4: equal weights, ssp2 2025 

In [2]:
# Rasters and weights for model 4 equal weights (ssp2 2025):

# Rasters
land_cover = './data/land_cover/land_cover_standardized.tif'
hsg = './data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif'
slope = './data/slope/slope_standardized.tif'
impairment = './data/watershed_impairment/impairment_standardized.tif'
mean_prcp = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif'
sd_prcp = './data/projected_precip/precip_ssp2_2025/ssp2_2025_standard_dev_prcp_rate_standarized.tif'
impervious = './data/percent_impervious/impervious_standardized.tif'
pop_density = './data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif'
svi = './data/svi/svi_standardized.tif'

# Weights
land_cover_weight = 0.1111
hsg_weight = 0.1111
slope_weight = 0.1111
impairment_weight = 0.1111
mean_prcp_weight = 0.1111
sd_prcp_weight = 0.1111
impervious_weight = 0.1111
pop_density_weight = 0.1111
svi_weight = 0.1111

in_rasters = [land_cover, 
              hsg, slope, 
              impairment, 
              mean_prcp, 
              sd_prcp, 
              impervious, 
              pop_density, 
              svi]

weights = [land_cover_weight, 
           hsg_weight, 
           slope_weight, 
           impairment_weight, 
           mean_prcp_weight, 
           sd_prcp_weight, 
           impervious_weight, 
           pop_density_weight, 
           svi_weight]

for raster, weight in zip(in_rasters, weights):
    print(raster, weight)

./data/land_cover/land_cover_standardized.tif 0.1111
./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif 0.1111
./data/slope/slope_standardized.tif 0.1111
./data/watershed_impairment/impairment_standardized.tif 0.1111
./data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif 0.1111
./data/projected_precip/precip_ssp2_2025/ssp2_2025_standard_dev_prcp_rate_standarized.tif 0.1111
./data/percent_impervious/impervious_standardized.tif 0.1111
./data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif 0.1111
./data/svi/svi_standardized.tif 0.1111


In [3]:
# Model 4 weighted overlay 

# Output
out_raster = './data/final_outputs/model4_eqial_weights_ssp2_2025.tif'

# Open all rasters
srcs = [rasterio.open(raster) for raster in in_rasters]

# Verify that there are the same number of srcs as there are weights
print(f'Lengh of srcs equals length of weights: {len(srcs) == len(weights)}')
for src, weight in zip(srcs, weights):
    print(src, weight)

# Copy metadata from src raster
meta = srcs[0].meta.copy()
nodata = srcs[0].nodata

meta.update(
    dtype = rasterio.float32,
    count = 1,
    nodata = nodata,
    tiled = True, 
    blockxsize = 128, 
    blockysize = 128,
    compress = 'deflate',
    predictor = 3,
    BIGTIFF = 'yes'
)

# Total blocks
total_blocks = sum(1 for _ in srcs[0].block_windows(1))
print(f'Total blocks {total_blocks}')

# Make output raster
with rasterio.open(out_raster, 'w', **meta) as dst:
    
    # For each block window
    for ji, window in tqdm(srcs[0].block_windows(1), total = total_blocks, desc = 'Block window processing'):
        # Create an empty array of zeros
        weighted_sum = np.zeros(
            (window.height, window.width),
            dtype = rasterio.float32)

        # Create an array to store all VALID (aka not nodata) cells (starts off with all cells as valid)
        valid_array = np.ones(
            (window.height, window.width),
            dtype = bool)
        
        # Go through each variable, weighs
        for src, weight in zip(srcs, weights):
            data = src.read(1, window = window)

            # Valid cells (where the cells do not equal nodata)
            valid_cells = data!= nodata

            # Update valid_array after checking for nodata cells
            valid_array = valid_array & valid_cells # Only True if the cell is actually valid

            # Replace nodata cells with 0 to not mess up with weighted sum calculations
            data = np.where(valid_cells, data, 0)

            # Multiply the cell value by the weight 
            weighted_sum += data*weight

        # Set any nodata cells to nodata in the final output raster
        weighted_sum = np.where(valid_array, weighted_sum, nodata)

        # Write out final raster
        dst.write(weighted_sum, 1, window = window)
            

# Close all input rasters
for src in srcs:
    src.close()

Lengh of srcs equals length of weights: True
<open DatasetReader name='./data/land_cover/land_cover_standardized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/SSURGO_raw/hsg/hsg_FINAL_composite_standardized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/slope/slope_standardized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/watershed_impairment/impairment_standardized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/projected_precip/precip_ssp2_2025/ssp2_2025_standard_dev_prcp_rate_standarized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/percent_impervious/impervious_standardized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/population_density/pop_ssp2_rcp45_2025/ssp2_2025_density_standardized.tif' mode='r'> 0.1111
<open DatasetReader name='./data/svi/svi_standardized.tif' mode='r'> 0.1111
Total blocks 910224


Block window processing: 100%|██████████| 910224/910224 [1:06:21<00:00, 228.63it/s]


In [4]:
# Model 4 output raster

out_raster = './data/final_outputs/model4_eqial_weights_ssp2_2025.tif'

with rasterio.open(out_raster, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [3]:
# Reclassify cells for Model 4

out_raster = './data/final_outputs/model4_eqial_weights_ssp2_2025.tif'
out_raster_reclassified = '/work/hdd/bfqp/cchan2/model4/model4_equal_weights_ssp2_2025_RECLASSIFIED.tif'

with rasterio.open(out_raster) as src:
    profile = src.profile.copy()
    nodata = src.nodata
    profile.update(
        dtype = rasterio.int16,
        nodata = nodata,
        #blockxsize = 128,
        #blockysize = 128,
        #tiled = True
    )

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    with rasterio.open(out_raster_reclassified, 'w', **profile) as dst:

        for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):

            data = src.read(1, window = window)
            data_copy = data.copy()

            # Reclassify:
                # [10-8] >> 5 (highly suitable)
            data_copy[(data <=10) & (data >=8)] = 5
            
                # (8-6] >> 4 (moderately suitable)
            data_copy[(data <8) & (data >=6)] = 4
            
                # (6-4}] >> 3 (marginally suitable)
            data_copy[(data <6) & (data >=4)] = 3

                # (4-2] >> 2 (very unsuitable)
            data_copy[(data <4) & (data >=2)] = 2

                # (2-0] >> 1 (unsuitable)
            data_copy[(data <2) & (data >=0)] = 1

            # Write out reclassified raster
            dst.write(data_copy, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [12:54<00:00, 1175.74it/s]


In [4]:
# Model 4 reclassified output raster

out_raster_reclassified = '/work/hdd/bfqp/cchan2/model4/model4_equal_weights_ssp2_2025_RECLASSIFIED.tif'

with rasterio.open(out_raster_reclassified, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso

In [5]:
# Calculate % of each final suitability category

out_raster_reclassified = '/work/hdd/bfqp/cchan2/model4/model4_equal_weights_ssp2_2025_RECLASSIFIED.tif'

total_1 = 0
total_2 = 0
total_3 = 0
total_4 = 0
total_5 = 0
total_nodata = 0

with rasterio.open(out_raster_reclassified) as src:

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
        data = src.read(1, window = window, masked = True)

        total_1 += np.count_nonzero(data == 1) # Total cells = 1
        total_2 += np.count_nonzero(data == 2) # Total cells = 2
        total_3 += np.count_nonzero(data == 3) # Total cells = 3
        total_4 += np.count_nonzero(data == 4) # Total cells = 4
        total_5 += np.count_nonzero(data == 5) # Total cells = 5

        total_nodata += np.sum(data.mask) # Calculates the total number of nodata cells

    all_cells = src.width * src.height

    all_NON_nodata_cells = all_cells - total_nodata

    percent_5 = total_5 / all_NON_nodata_cells
    print(f'% highly suitable (5): {percent_5}')
    
    percent_4 = total_4 / all_NON_nodata_cells
    print(f'% moderately suitable (4): {percent_4}')

    percent_3 = total_3 / all_NON_nodata_cells
    print(f'% marginally suitable (3): {percent_3}')

    percent_2 = total_2 / all_NON_nodata_cells
    print(f'% very unsuitable (2): {percent_2}')

    percent_1 = total_1 / all_NON_nodata_cells
    print(f'% absolutely unsuitable (1): {percent_1}')

    print(f'%s add to : {percent_1 + percent_2 + percent_3 + percent_4 + percent_5}')

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:15<00:00, 2882.99it/s]


% highly suitable (5): 7.195043357629339e-05
% moderately suitable (4): 0.014747305251047244
% marginally suitable (3): 0.4031065081467361
% very unsuitable (2): 0.5725743234294708
% absolutely unsuitable (1): 0.009499912739169615
%s add to : 1.0


## Change detection btwn current and future conditions 

### SSP2 2050 compared to SSP2 2025

In [8]:
# Subtract present conditions from future conditions to calculate change
    # 0 = no change
    # positive values = increase in suitability
    # negative values = decrease in suitability 

current_conditions = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif' # current
future_conditions = './data/final_outputs/model2_ahp_weights_ssp2_2050_RECLASSIFIED.tif' # present

change_detection = '/work/hdd/bfqp/cchan2/change_detection/ssp2_2050_ssp2_2025_change_detection.tif'


with rasterio.open(current_conditions) as src1:
    profile = src1.profile.copy()
    nodata = src1.nodata

    # Total number of blocks
    total_blocks = sum(1 for _ in src1.block_windows(1))
    print(f'Total blocks: {total_blocks}')
    
    with rasterio.open(future_conditions) as src2:

        with rasterio.open(change_detection, 'w', **profile) as dst:

            for ji, window in tqdm(src1.block_windows(1), total = total_blocks, desc = 'Block window processing'):
                current = src1.read(1, window = window) # Current conditions data
                future = src2.read(1, window = window) # Future conditions data

                diff = future - current # Difference btwn future and current conditions

                nodata_cells = (current == nodata) | (future == nodata) # Isolate nodata cells
                diff[nodata_cells] = nodata # Replace nodata cells with nodata value

                change = np.where(diff >0, 1,
                                  np.where(diff <0, -1, diff)) # Positive difference = 1; Negative difference = -1, No difference = 0
                change[nodata_cells] = nodata
                                

                dst.write(change, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [09:49<00:00, 1545.00it/s]


In [9]:
# Output

change_detection = '/work/hdd/bfqp/cchan2/change_detection/ssp2_2050_ssp2_2025_change_detection.tif'

with rasterio.open(change_detection, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso

### SSP5 2050 compared to SSP2 2025

In [10]:
# Subtract present conditions from future conditions to calculate change
    # 0 = no change
    # positive values = increase in suitability
    # negative values = decrease in suitability 

current_conditions = './data/final_outputs/model1_ahp_weights_ssp2_2025_RECLASSIFIED_2.tif' # current
future_conditions = './data/final_outputs/model3_ahp_weights_ssp5_2050_RECLASSIFIED.tif' # present

change_detection = '/work/hdd/bfqp/cchan2/change_detection/ssp5_2050_ssp2_2025_change_detection.tif'


with rasterio.open(current_conditions) as src1:
    profile = src1.profile.copy()
    nodata = src1.nodata

    # Total number of blocks
    total_blocks = sum(1 for _ in src1.block_windows(1))
    print(f'Total blocks: {total_blocks}')
    
    with rasterio.open(future_conditions) as src2:

        with rasterio.open(change_detection, 'w', **profile) as dst:

            for ji, window in tqdm(src1.block_windows(1), total = total_blocks, desc = 'Block window processing'):
                current = src1.read(1, window = window) # Current conditions data
                future = src2.read(1, window = window) # Future conditions data

                diff = future - current # Difference btwn future and current conditions

                nodata_cells = (current == nodata) | (future == nodata) # Isolate nodata cells
                diff[nodata_cells] = nodata # Replace nodata cells with nodata value

                change = np.where(diff >0, 1,
                                  np.where(diff <0, -1, diff)) # Positive difference = 1; Negative difference = -1, No difference = 0
                change[nodata_cells] = nodata
                                

                dst.write(change, 1, window = window)

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [07:56<00:00, 1910.92it/s]


In [11]:
# Output

change_detection = '/work/hdd/bfqp/cchan2/change_detection/ssp5_2050_ssp2_2025_change_detection.tif'

with rasterio.open(change_detection, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Reso